We use the stable baselines library to train a sokoban solver. However we did go through some trial and error to find the right model to use.

First of all, we initially intended to use Deep Q-Learning (DQN) to train our model. However when doing so, we were not getting good results.
The reason has to do with the way we implemented the input of the game. DQN works well if the action space is concise, such as [Up, Down, Left, Right].

However our input was the [Up, Down, Left, Right] for every box. The larger action space made Proximal Policy Optimization (PPO) make more sense. 

The multiple maps to use in order are:

- Easy: 'sokoban05a.txt'
- Medium: 'sokoban05c.txt'
- Hard: 'sokoban05b.txt'

In [2]:
import gymnasium as gym
from gymnasium import spaces
from SKBGame import SKBGame
from SKBView import SKBView
import numpy as np

seed=1234
sbkmap = 'sokoban05a.txt'

class SKBEnv(gym.Env):
    def __init__(self, skb_data):
        super(SKBEnv, self).__init__()
        
        # Initialize game
        self.game = SKBGame(skb_data)
        self._data = skb_data

        # Define action space (number of available moves)
        self.action_space = spaces.Discrete(len(self.game.choices_all))

        # Define observation space (state representation)
        self.observation_space = spaces.Box(low=0, high=4, shape=(self.game.map.size.bounds[0], self.game.map.size.bounds[1]), dtype=np.float32)
        
    def reset(self, seed=None, options=None):
        """Resets the game state."""
        super().reset(seed=seed)
        self.game = SKBGame(self._data)
        return np.array(self._get_observation(), dtype=np.float32), {}

    def step(self, action):
        """Executes an action and returns state, reward, done flag."""
        new_game, valid = self.game.pick_all(action)
        self.game = new_game

        # Compute reward
        reward = -(sum(self.game.heuristics.close) ** 3)
        reward += -(999999999 if any(self.game.heuristics.dead) else 0)
        
        if not valid:
            reward += -90000
            
        win_reward = 100000 if self.game.win() == 1 else -30000 if self.game.win() == 2 else -50000
        reward += win_reward
        
        # Check if game is finished
        done = self.game.win() in [1, 2]

        return np.array(self._get_observation(), dtype=np.float32), reward, done, False, {}

    def _get_observation(self):
        """Encodes the current game state as a numerical observation."""
        char_map = {'_': 0, 'W': 1, 'B': 2, 'S': 3, 'P':4}
        return np.array([[char_map[piece] for piece in row] for row in self.game.map._skbm_grid], dtype=np.int32)

    def render(self):
        """Displays the current game state."""
        return self.game.display()

In [3]:
env = SKBEnv(sbkmap)

from stable_baselines3 import PPO
from tqdm import tqdm
import torch

model = PPO("MlpPolicy", env, verbose=0, device='cuda',
            batch_size=1024,
            learning_rate=1e-4,
            ent_coef=0.01,
            gamma=0.98,
            n_steps=1024)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

total_steps = 3
with tqdm(total=total_steps) as pbar:
    for step in range(total_steps):
        model.learn(total_timesteps=10000)
        pbar.update(1)

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [03:46<00:00, 75.38s/it]


This will display all the intermediate attempts from the PPO.

In [4]:
from IPython.display import clear_output

max_attempts = 1000
attempts = 0
while attempts < max_attempts:
    clear_output(wait=True)
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, _, _ = env.step(action)
    print(attempts)
    env.render()  # Display AI's moves
    if env.game.win() == 1:
        break
    attempts += 1
print(f"AI completed after {attempts} attempts")

32
############
#   #      #
#   # #    #
#          #
###        #
#     ######
#     @$$###
############
AI completed after 32 attempts


If you want to play the animation of the final solution, run this block.

In [5]:
history = env.game.history()
    
SKBView(sbkmap, history).play(0.03)

############
#   #      #
#   # #    #
#          #
###        #
#     ######
#     @$$###
############
